In [ ]:
import os
import json
import time
import re
import urllib.parse
import feedparser
import pandas as pd
import httpx
from datetime import datetime, timedelta
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from dotenv import load_dotenv
from kiwipiepy import Kiwi

# ==========================================
# 1. 환경 설정 및 전역 변수
# ==========================================
env_path = Path.cwd().parent / ".env"
if not env_path.exists():
    env_path = Path.cwd() / ".env"
load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")
http_client = httpx.Client(verify=False, timeout=httpx.Timeout(300.0, connect=60.0))
client = OpenAI(api_key=api_key, http_client=http_client)
kiwi = Kiwi(model_type='cong')

class TokenTracker:
    def __init__(self):
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.price_input = 0.15 / 1000000
        self.price_output = 0.60 / 1000000
        self.exchange_rate = 1350

    def add(self, usage):
        self.total_input_tokens += usage.prompt_tokens
        self.total_output_tokens += usage.completion_tokens

    def get_report(self):
        cost_usd = (self.total_input_tokens * self.price_input) + (self.total_output_tokens * self.price_output)
        cost_krw = cost_usd * self.exchange_rate
        return f"""
[GPT API 사용량 보고서]
- 입력 토큰: {self.total_input_tokens:,} tokens
- 출력 토큰: {self.total_output_tokens:,} tokens
- 예상 비용: ${cost_usd:.4f} (약 {cost_krw:.2f}원)
"""

tracker = TokenTracker()


ISSUE_WEIGHT_MAP = {
    '참사': 200, '압수수색': 200, '마비': 200, '먹통': 200, '탄핵': 200,
    '수사': 150, '의혹': 150, '은폐': 150, '조작': 150, '유출': 150, 
    '늑장': 150, '결손': 150, '낙하산': 150,
    '비판': 120, '논란': 120, '부실': 120, '지적': 100, '파행': 100, 
    '특혜': 100, '탁상': 100, '낭비': 100,
    '우려': 80, '사고': 70, '갈등': 60, '공방': 50, '지연': 50
    
}

# [복구] 150개 키워드 데이터 전체 보존
KEYWORDS_DATA = [
    ("경찰관직무집행법", "경찰 제도 및 행정"), ("경찰공무원법", "경찰 제도 및 행정"), ("지방교부세", "지방재정 및 공기업"),
    ("지방자치법", "지방자치 제도 및 분권"), ("재정분권", "지방재정 및 공기업"), ("간토대학살사건", "강제동원 및 피해 지원"),
    ("경찰청예산", "경찰 제도 및 행정"), ("지방공기업법", "지방재정 및 공기업"), ("지방재정법", "지방재정 및 공기업"),
    ("지방자치법개정", "지방자치 제도 및 분권"), ("이태원참사", "대형 참사 및 특별법"), ("자치경찰제", "경찰 제도 및 행정"),
    ("지방세법", "지방세 및 세제"), ("특례시", "지방자치 제도 및 분권"), ("재난지원금", "재난지원 및 구호"),
    ("지방시대위원회", "지방자치 제도 및 분권"), ("지방선거", "지방자치 제도 및 분권"), ("경찰인사", "경찰 제도 및 행정"),
    ("경찰수사권조정", "경찰 제도 및 행정"), ("지방출자출연기관", "지방재정 및 공기업"), ("재난관리기본법", "재난관리 및 법령"),
    ("소방청예산", "소방 조직 및 인사"), ("소방인력보충", "소방 조직 및 인사"), ("디지털정부", "디지털 정부 및 정보화"),
    ("정부조직법", "정부조직 및 개편"), ("공무원처우개선", "지방공무원 인사 및 처우"), ("지방자치단체", "지방자치 제도 및 분권"),
    ("제주43사건", "제주 4·3 및 여순 사건"), ("화재예방", "화재예방 및 조사"), ("지역사랑상품권", "지역사랑상품권"),
    ("청년일자리", "일자리 및 청년 정책"), ("경찰개혁", "경찰 제도 및 행정"), ("경찰국설치", "경찰 제도 및 행정"),
    ("국가경찰", "경찰 제도 및 행정"), ("자치경찰", "경찰 제도 및 행정"), ("고향사랑기부금", "고향사랑기부제"),
    ("지방세특례제한법", "지방세 및 세제"), ("재난대응", "재난대응 및 복구"), ("특별재난지역", "재난대응 및 복구"),
    ("지방소멸", "인구 정책 및 균형발전"), ("지역균형발전", "인구 정책 및 균형발전"), ("中央지방협력회의", "지방자치 제도 및 분권"),
    ("공직자윤리법", "행정혁신 및 적극행정"), ("정보공개법", "정보공개 및 기록 관리"), ("주민등록법", "민원 및 행정 서비스"),
    ("민원처리에관한법률", "민원 및 행정 서비스"), ("적극행정", "행정혁신 및 적극행정"), ("정부혁신", "행정혁신 및 적극행정"),
    ("국가안전시스템", "재난관리 및 법령"), ("다중이용시설안전", "생활 및 다중시설 안전"), ("어린이보호구역", "어린이 및 안전취약계층"),
    ("보행자안전", "교통 및 보행 안전"), ("개인정보보호", "디지털 정부 및 정보화"), ("클라우드컴퓨팅", "디지털 정부 및 정보화"),
    ("공공데이터", "데이터 및 인공지능"), ("민주화운동", "민주화 운동"), ("여순사건", "제주 4·3 및 여순 사건"),
    ("지방채", "지방재정 및 공기업"), ("공공기관통합", "지방재정 및 공기업"), ("새마을운동", "사회단체 지원"),
    ("자원봉사", "사회단체 지원"), ("지방자치단체합동평가", "지방자치 제도 및 분권"), ("상훈법", "상훈 및 국가 상징"),
    ("민방위", "국가 안보 및 민방위"), ("을지연습", "국가 안보 및 민방위"), ("풍수해", "자연재난 대응"),
    ("폭염대책", "자연재난 대응"), ("지진대응", "자연재난 대응"), ("기부금품법", "사회단체 지원"),
    ("주민투표", "주민자치 및 주민참여"), ("주민소환", "주민자치 및 주민참여"), ("주민참여예산", "주민자치 및 주민참여"),
    ("지방행정체제개편", "특별자치도 및 구역개편"), ("행정구역통합", "특별자치도 및 구역개편"), ("세종특별자치시", "특별자치도 및 구역개편"),
    ("강원특별자치도", "특별자치도 및 구역개편"), ("전북특별자치도", "특별자치도 및 구역개편"), ("지방공무원법", "지방공무원 인사 및 처우"),
    ("지방공무원교육", "지방공무원 인사 및 처우"), ("지방자치인재개발원", "지방공무원 인사 및 처우"), ("지방공공기관경평", "지방재정 및 공기업"),
    ("지역화폐", "지역경제 및 지역화폐"), ("전통시장활성화", "지역경제 및 지역화폐"), ("소상공인지원", "지역경제 및 지역화폐"),
    ("디지털플랫폼정부", "디지털 정부 및 정보화"), ("모바일신분증", "디지털 정부 및 정보화"), ("행정망마비", "디지털 정부 및 정보화"),
    ("사이버보안", "디지털 정부 및 정보화"), ("재해구호", "재난지원 및 구호"), ("재난심리회복", "재난지원 및 구호"),
    ("화재안전조사", "화재예방 및 조사"), ("소방시설법", "소방 시설 및 안전"), ("구급대원처우", "소방 조직 및 인사"),
    ("소방장비현대화", "소방 장비 및 시스템"), ("산불대응", "자연재난 대응"), ("침수대책", "자연재난 대응"),
    ("지방세체납", "지방세 및 세제"), ("고액상습체납자", "지방세 및 세제"), ("취득세", "지방세 및 세제"),
    ("재산세", "지방세 및 세제"), ("법인지방소득세", "지방세 및 세제"), ("지역안전지수", "재난관리 및 법령"),
    ("안전신문고", "재난관리 및 법령"), ("국가상징", "상훈 및 국가 상징"), ("국기법", "상훈 및 국가 상징"),
    ("정부청사관리", "정부조직 및 개편"), ("책임운영기관", "정부조직 및 개편"), ("정부혁신어벤져스", "행정혁신 및 적극행정"),
    ("적극행정위원회", "행정혁신 및 적극행정"), ("지방소멸대응기금", "인구 정책 및 균형발전"), ("인구감소지역", "인구 정책 및 균형발전"),
    ("생활인구", "인구 정책 및 균형발전"), ("귀농귀촌", "인구 정책 및 균형발전"), ("지역사회혁신", "지역사회 및 마을공동체"),
    ("마을기업", "지역사회 및 마을공동체"), ("청년마을", "지역사회 및 마을공동체"), ("지역공동체활성화", "지역사회 및 마을공동체"),
    ("민원공무원보호", "민원 및 행정 서비스"), ("악성민원대응", "민원 및 행정 서비스"), ("원스톱민원창구", "민원 및 행정 서비스"),
    ("디지털민원", "민원 및 행정 서비스"), ("전자정부법", "디지털 정부 및 정보화"), ("빅데이터활용", "데이터 및 인공지능"),
    ("인공지능행정", "데이터 및 인공지능"), ("국가기록원", "정보공개 및 기록 관리"), ("기록관리시스템", "정보공개 및 기록 관리"),
    ("과거사위원회", "과거사 진상 규명"), ("대일항쟁기위원회", "강제동원 및 피해 지원"), ("민주화운동기념사업회", "민주화 운동"),
    ("바르게살기운동", "사회단체 지원"), ("자유총연맹", "사회단체 지원"), ("지방행정연구원", "지방자치 제도 및 분권"),
    ("지방세연구원", "지방세 및 세제"), ("지방재정공제회", "지방재정 및 공기업"), ("대한지방행정공제회", "지방재정 및 공기업"),
    ("소방공무원법", "소방 조직 및 인사"), ("소방산업육성", "소방 장비 및 시스템"), ("지능형CCTV", "재난관리 및 법령"),
    ("재난안전통신망", "재난관리 및 법령"), ("국가안전대진단", "재난관리 및 법령"), ("지하차도침수", "자연재난 대응"),
    ("지하시설물물막이판", "자연재난 대응"), ("폭설대응", "자연재난 대응"), ("가뭄대책", "자연재난 대응"),
    ("지방공공요금", "지방재정 및 공기업"), ("물가안정대책", "지역경제 및 지역화폐"), ("청년창업지원", "일자리 및 청년 정책")
]

BASE_DIR = r"C:\sw\news"
os.makedirs(BASE_DIR, exist_ok=True)
BATCH_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
JSON_PATH = os.path.join(BASE_DIR, f"행안부_보고서_{BATCH_ID}.json")
CSV_PATH = os.path.join(BASE_DIR, f"행안부_데이터_{BATCH_ID}.csv")

# ==========================================
# 3. 데이터 수집 엔진 (공격형 쿼리 강화 버전)
# ==========================================

def fetch_massive_news(args):
    skw, mkw = args
    
    # [전략적 수정] 키워드 뒤에 '부정적 트리거'를 조합하여 검색 품질 극대화
    # 구글 뉴스에서 '논란, 부실, 의혹, 비판, 사고, 마비' 중 하나라도 걸린 뉴스만 수집
    negative_triggers = "(논란 OR 부실 OR 의혹 OR 비판 OR 사고 OR 마비 OR 참사 OR 늑장 OR 방치)"
    search_query = f"{skw} {negative_triggers}"
    
    query = urllib.parse.quote(search_query)
    rss_url = f"https://news.google.com/rss/search?q={query}&hl=ko&gl=KR&ceid=KR:ko"
    
    try:
        # 타임아웃 처리를 위해 httpx 사용 권장 (feedparser의 느린 속도 보완)
        feed = feedparser.parse(rss_url)
        items, cutoff = [], datetime.now() - timedelta(days=14)
        
        for entry in feed.entries[:50]: 
            try:
                pub_date = datetime(*entry.published_parsed[:6])
                if pub_date >= cutoff:
                    title = entry.title.split(' - ')[0]
                    
                    # [필터링 강화] 제목에 최소한의 행안부 관련 단어가 있는지 재확인 (노이즈 제거)
                    score = sum(val for word, val in ISSUE_WEIGHT_MAP.items() if word in title)
                    
                    items.append({
                        "title": title, 
                        "category": mkw, 
                        "keyword": skw, 
                        "score": score,
                        "pub_date": pub_date.strftime('%Y-%m-%d')
                    })
            except: continue
        return items
    except: return []
# 1. 후보군을 40개로 확장 (ai_massive_pre_filter)
def ai_massive_pre_filter(news_batch):
    indexed_titles = "\n".join([f"{i}. {n['title']}" for i, n in enumerate(news_batch)])
    prompt = f"""
당신은 국회 행정안전위원회 소속 '야당 저격수' 국회의원의 수석 보좌관입니다. 
당신의 목표는 이번 상임위에서 행정안전부 장관을 정책적으로 강하게 압박하여 사과를 받아내거나 제도 개선 약속을 받아내는 것입니다.

제공된 뉴스 리스트에서 '장관에게 치명적인 공격 쟁점'이 될 뉴스 **반드시 40개**를 고르십시오. (부족하더라도 최대한 40개에 가깝게 추출)

[선별 원칙]
1. 단순 지자체 홍보, 미담, 행사, 수상 소식은 '쓰레기 데이터'로 간주하고 무조건 버릴 것.
2. 예산 낭비, 인명 사고의 책임 소재, 공직자의 윤리적 결함, 시스템 마비와 관련된 기사를 최우선할 것.
3. '동작구', '아산시' 등 특정 기초지자체의 소소한 민원 해결 기사는 행안위 쟁점이 아니므로 제외할 것.
4. "부실", "의혹", "비판", "참사", "마비" 키워드가 없더라도, 기사 이면에 정부의 관리 소홀이 숨겨진 것을 찾아낼 것. 

[뉴스 리스트]
{indexed_titles}

반드시 선택한 뉴스의 '숫자'만 쉼표로 구분하여 출력하십시오.
"""
    try:
        res = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0, seed=42
        )
        tracker.add(res.usage)
        indices = [int(s) for s in re.findall(r'\d+', res.choices[0].message.content)]
        return [news_batch[i] for i in indices if i < len(news_batch)]
    except Exception as e:
        print(f"⚠️ AI 필터링 오류: {e}")
        return news_batch[:40]


# 2. 10~13개 개수 강제 및 다양성 확보 (generate_consolidated_strategy)
def generate_consolidated_strategy(news_list):
    if not news_list: return []
    context = "\n".join([f"- [{n['category']}] {n['title']}" for n in news_list])
    
    prompt = f"""
당신은 국회 행정안전위원회 소속 '야당 저격수' 국회의원의 수석 보좌관입니다. 
제공된 뉴스를 바탕으로 장관을 상대로 상임위에서 마이크를 잡을 수 있는 [심층 현안 분석 리포트]를 작성하십시오.

### [회의록 패턴 학습 데이터] ###
- 질의(공격): "사전 인지 여부", "절차적 정당성 미비", "현장 이행력 부실", "반복되는 사고에 대한 행정 방치"를 근거로 압박.
- 답변(방어): "사실관계 확인", "즉시 조치 사항", "범부처 협력 체계", "근본적 매뉴얼 정비 및 제도 개선" 순으로 논리 전개.

### [작성 지침 - 중복 방지 및 개수 확보] ###
1. **중복 절대 금지 (1사건 1현안)**: 동일한 사건(예: 특정 기업의 정보유출, 특정 지역의 참사 등)을 제목만 바꿔서 여러 개의 항목으로 만들지 마십시오. 발견 시 감점 요인입니다. 반드시 '사건 단위'로 통합하십시오.
2. **분량 강제 및 다양성 확보**: 최종 리스트는 **반드시 10개에서 13개 사이**로 구성하십시오.
   - 만약 '정치적 대형 쟁점'이 10개 미만이라면, **'행정망 보안', '지방 소멸 대책', '소방 공무원 처우', '민원 제도 개선', '공직 기강 점검'** 등 행안부 소관 실무 영역 중 비판적인 시각이 있는 뉴스를 추가하여 **중복 없이** 개수를 채우십시오.
3. **keyword**: 현안의 분류와 본질을 관통하는 명확한 제목을 작성하되, 앞에 반드시 [분류]를 명시하십시오. (예: [공직기강] 재해구호협회 채용 비리 의혹 및 행안부 관리 부실)
4. **background**: 단순히 사건 요약이 아니라 '누가 어떤 잘못을 해서 어떤 논란이 생겼는지'를 **반드시 4~5개의 구체적인 명사형 개조식 문장**으로 상세히 기술하십시오.
5. **question (명사형 종결)**: 장관의 책임을 묻는 핵심 질의 항목 **2개**를 도출하십시오. (~입니까? 금지)
6. **answer (내용 풍부화 및 명사형 종결)**: 위 질문 2개에 각각 직접 대응하는 행안부의 정책적 대응 논리와 구체적 조치를 작성하십시오. 정부의 방어 논리(사실확인, 매뉴얼 정비 등)와 함께 반드시 '구체적 제도 개선안'을 명사형으로 3개 이상 포함하십시오.(질문당 2~3개, 총 4~6개 답변 유지) -답변 작성 시 유의사항: 모든 현안의 답변은 해당 사건의 **특수성(구체적 지명, 기관명, 사고 유형)**을 반영하여 개별적으로 작성하십시오. '전담 기구 신설'이나 '매뉴얼 정비'와 같은 상투적인 표현만으로 답변 리스트를 채우는 행위를 엄격히 금지하며, 발견 시 중복 데이터로 간주합니다."


[뉴스 리스트]
{context}

반드시 아래 JSON 리스트 형식으로 답변하십시오.
[
  {{
    "keyword": "[분류] 현안 통합 제목",
    "background": "- 상세내용1\\n- 상세내용2\\n- 상세내용3\\n- 상세내용4\\n- 상세내용5",
    "question": "- 질의 쟁점 1\\n- 질의 쟁점 2",
    "answer": "- 질문1 대응정책\\n- 질문1 구체조치\\n- 질문2 대응정책\\n- 질문2 구체조치\\n- 질문2 향후계획"
  }}
]
"""
    
    try:
        res = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "행안부 국회 대응 전문가입니다. 풍부한 배경 설명과 다각적인 정책 대책을 포함한 명사형 개조식 분석을 수행합니다."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )
        tracker.add(res.usage)
        raw_content = res.choices[0].message.content
        cleaned_json = re.sub(r'```json|```', '', raw_content).strip()
        data = json.loads(cleaned_json)
        return data if isinstance(data, list) else [data]
    except Exception as e: 
        print(f"⚠️ 분석 실패: {e}")
        return []

# ==========================================
# 5. 메인 파이프라인
# ==========================================

def main():
    start_time = time.time()
    
    # (1) 대규모 수집
    step1_start = time.time()
    print(f"🚀 [1/4] 뉴스 수집 엔진 가동 (키워드 {len(KEYWORDS_DATA)}개)")
    raw_pool = []
    with ThreadPoolExecutor(max_workers=60) as exec:
        futures = [exec.submit(fetch_massive_news, kw) for kw in KEYWORDS_DATA]
        for f in as_completed(futures): raw_pool.extend(f.result())
    
    df = pd.DataFrame(raw_pool).drop_duplicates(subset=['title'])
    print(f"✅ [1단계 완료] {len(df)}건 확보. ({time.time()-step1_start:.1f}초)")

    # (2) 형태소 분석 및 전수 필터링
    step2_start = time.time()
    print(f"🚀 [2/4] 형태소 빈도 분석 및 AI 쟁점 선별")
    def get_ns(t):
        r = kiwi.analyze(re.sub(r'[^가-힣]', ' ', t))
        return {tk.form for m in r for tk in m[0] if tk.tag in ['NNG', 'NNP'] and len(tk.form) > 1}
    
    df['ns'] = df['title'].apply(get_ns)
    all_words = [w for s in df['ns'] for w in s]
    top_words = {k[0] for k in Counter(all_words).most_common(100)}
    df['freq_score'] = df['ns'].apply(lambda x: len(x & top_words))
    df['total_score'] = df['freq_score'] + (df['score'] * 2.5)
    
    candidates = df.sort_values(by='total_score', ascending=False).head(200).to_dict('records')
    final_30_raw = ai_massive_pre_filter(candidates)
    
    seen = set()
    final_30 = []
    for n in final_30_raw:
        if n['title'] not in seen:
            final_30.append(n)
            seen.add(n['title'])
    print(f"✅ [2단계 완료] 핵심 쟁점 후보 30건 확정. ({time.time()-step2_start:.1f}초)")

    # (3) 지능형 통합 분석 (신규 지능형 레이어)
    step3_start = time.time()
    print(f"🚀 [3/4] 지능형 전략 레이어 가동 (회의록 논리 기반 분석 중)...")
    all_issues = generate_consolidated_strategy(final_30)
    
    for issue in all_issues:
        issue['batch_id'] = BATCH_ID
        issue['created_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    print(f"✅ [3단계 완료] 총 {len(all_issues)}개의 전략 리포트 생성 완료. ({time.time()-step3_start:.1f}초)")

    # (4) 저장
    step4_start = time.time()
    if all_issues:
        final_df = pd.DataFrame(all_issues)
        cols = ['batch_id', 'created_at', 'keyword', 'background', 'question', 'answer']
        for c in cols:
            if c not in final_df.columns: final_df[c] = ""
        final_df = final_df[cols]

        final_df.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')
        final_df.to_json(JSON_PATH, orient='records', force_ascii=False, indent=4)
        print(f"💾 결과 저장 완료: {CSV_PATH}")
    else:
        print("⚠️ 생성된 분석 데이터가 없습니다.")

    print("-" * 50)
    print(f"✨ 모든 공정 완료! (총 소요시간: {time.time()-start_time:.1f}초)")
    print(tracker.get_report())

if __name__ == "__main__":
    main()